# A cube-toss video with MitsubaVideo

Create a small animation, compare camera views, render each frame, and encode a video. The cube follows a prescribed ballistic arc without wear maps or external images. It is a rendering example, not a collision simulator.

Install `packages/mitsuba_video[usd,notebook]` from the repository root and select that environment's kernel. FFmpeg is needed for the video step. The camera section follows Mitsuba's [multi-view tutorial](https://mitsuba.readthedocs.io/en/stable/src/rendering/multi_view_rendering.html), adapted to this application's frame sequence.

Rendering uses a subprocess so Windows Mitsuba shutdown cannot terminate the notebook kernel. Output stays under `output/notebook/`.

In [ ]:
import importlib.util
import json
import shutil
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, Video, display

root = Path.cwd().resolve()
if not (root / "packages/mitsuba_video").is_dir():
    root = root.parent
assert (root / "packages/mitsuba_video").is_dir(), "Open this notebook from the repository"
out = root / "output/notebook"
out.mkdir(parents=True, exist_ok=True)

def cli(*args):
    result = subprocess.run([sys.executable, "-m", "mitsuba_video", *map(str, args)],
                            cwd=root, capture_output=True, text=True, timeout=600)
    print(result.stdout)
    if result.returncode:
        raise RuntimeError(result.stderr)
    return result

cli("doctor")

## Generate the animation

The fixture creates a rotating cube with a plain blue material. Every pose becomes one video frame. No wear atlas is generated.

In [ ]:
demo = out / "cube"
cli("fixture", demo, "--kind", "cube-throw", "--frames", "12")
cli("validate", demo)
cli("inspect", demo)

## Render one frame

`--frames -1:` selects the final input frame. `beauty/` contains the 3D image; `frames/` contains the annotated image used by the video encoder. The small resolution and sample count keep this tutorial quick.

In [ ]:
single = out / "single"
cli("render", demo, "--out", single, "--frames", "6:7", "--res", "320x180",
    "--spp", "8", "--look", "plain", "--no-panel", "--no-chart", "--no-encode")
display(Image(filename=str(single / "beauty/frame_000000.png")))

## Render the same frame from multiple views

The upstream tutorial creates several sensors around a scene. Here, each azimuth renders the same simulation frame to its own folder. All views keep the same exposure, samples and resolution. `--camera orbit` changes the viewpoint over an entire video, while `track` follows the moving body.

In [ ]:
from PIL import Image as PILImage

views = []
for azimuth in (-60, 0, 60):
    target = out / f"view-{azimuth}"
    cli("render", demo, "--out", target, "--frames", "6:7", "--azimuth", azimuth,
        "--elevation", "25", "--res", "240x135", "--spp", "8", "--look", "plain",
        "--no-panel", "--no-chart", "--no-encode")
    views.append(PILImage.open(target / "beauty/frame_000000.png").convert("RGB"))
grid = PILImage.new("RGB", (720, 135))
for i, image in enumerate(views):
    grid.paste(image, (i * 240, 0))
display(grid)

cli("render", demo, "--out", out / "orbit", "--camera", "orbit", "--orbit-degrees", "180",
    "--res", "160x90", "--spp", "4", "--look", "plain", "--no-panel", "--no-chart", "--no-encode")

## Render the sequence and resume

Each selected simulation frame produces a PNG. Increase samples and resolution after checking the camera and wear appearance. Resume checks render settings; keep the input files unchanged.

In [ ]:
sequence = out / "sequence"
args = ("render", demo, "--out", sequence, "--res", "160x90", "--spp", "4",
        "--look", "plain", "--no-panel", "--no-chart", "--no-encode")
cli(*args)
config = json.loads((sequence / "render_config.json").read_text())
print("Backend:", config["mitsuba"])
print("Step timings:", config.get("timing_s_per_frame"))
cli(*args, "--resume")
print("Frames:", len(list((sequence / "frames").glob("frame_*.png"))))

## Compare surface shading

PBR uses the material values; diffuse removes the specular lobe; metallic forces metal. AO casts hemisphere rays to show geometric occlusion. These controls work without a scalar field.

In [ ]:
for shading in ("diffuse", "metallic", "ao"):
    target = out / shading
    cli("render", demo, "--out", target, "--frames", "-1:", "--res", "160x90",
        "--spp", "4", "--look", "plain", "--shading", shading, "--no-encode")
    display(Image(filename=str(target / "beauty/frame_000000.png")))

## Encode without rendering again

FFmpeg reads the annotated PNG sequence. Twelve frames span 0.6 seconds in this fixture; 20 fps gives approximately the same playback duration. Changing fps does not change the time labels.

In [ ]:
if shutil.which("ffmpeg"):
    cli("encode", sequence, "--name", "tutorial", "--fps", "20", "--hold", "0")
    display(Video(filename=str(sequence / "tutorial.mp4"), embed=True))
else:
    print("Install FFmpeg, then rerun this cell to encode the video.")

## Write a dense mesh and try minimal displacement

A height map can move existing vertices. This example generates its own grid and height image, with displacement disabled in the first render and enabled in the second. The scale is in world units and the formula is `scale * (height - 0.5)`. There is no subdivision or animated displacement. Poses use `(w,x,y,z)` quaternions.

In [ ]:
import numpy as np
from PIL import Image as PILImage
from mitsuba_video.sources.manifest.writer import ManifestWriter

writer = ManifestWriter(out / "height-grid", name="Minimal height displacement")
u, v = np.meshgrid(np.linspace(0, 1, 25), np.linspace(0, 1, 25))
vertices = np.column_stack((2 * u.ravel() - 1, np.zeros(u.size), 2 * v.ravel() - 1)).astype("float32")
uvs = np.column_stack((u.ravel(), v.ravel())).astype("float32")
normals = np.tile(np.float32([0, 1, 0]), (u.size, 1))
faces = []
for row in range(24):
    for col in range(24):
        a = row * 25 + col
        faces.extend(((a, a + 26, a + 1), (a, a + 25, a + 26)))
yy, xx = np.mgrid[:64, :64] / 63
height = (0.5 + 0.45 * np.sin(4 * np.pi * xx) * np.sin(4 * np.pi * yy)).astype("float32")
writer.write_image("height.png", height)
writer.add_body(0, "grid", vertices, normals, uvs, np.uint32(faces), fixed=True,
                material={"base": {"albedo": [0.07, 0.3, 0.5], "roughness": 0.45, "height": "height.png"}})
writer.add_frame(0, [[0, 0, 0]], [[1, 0, 0, 0]])
manifest = writer.close()
for scale in (0, 0.15):
    target = out / f"height-{scale}"
    cli("render", manifest, "--out", target, "--res", "240x135", "--spp", "8", "--look", "plain",
        "--displacement-scale", scale, "--no-panel", "--no-chart", "--no-encode")
    display(Image(filename=str(target / "beauty/frame_000000.png")))

## Compare interpolated and face normals

Mitsuba's OBJ `face_normals=true` discards vertex normals and produces a faceted appearance. The application exposes this as `--face-normals`. This low-poly octahedron uses outward vertex normals for the smooth comparison and geometric face normals for the faceted render. No bunny asset needs downloading.

In [ ]:
writer = ManifestWriter(out / "facets", name="Face-normal comparison")
vertices = np.float32([[1,0,0], [-1,0,0], [0,1,0], [0,-1,0], [0,0,1], [0,0,-1]])
faces = np.uint32([[2,4,0], [2,1,4], [2,5,1], [2,0,5], [3,0,4], [3,4,1], [3,1,5], [3,5,0]])
writer.add_body(0, "octahedron", vertices, vertices.copy(), np.zeros((6, 2), np.float32), faces,
                material={"base": {"albedo": [0.08, 0.3, 0.5], "roughness": 0.5}})
writer.add_frame(0, [[0, 0, 0]], [[1, 0, 0, 0]])
manifest = writer.close()
for flag in ("--no-face-normals", "--face-normals"):
    target = out / flag.lstrip("-")
    cli("render", manifest, "--out", target, flag, "--normal-strength", "0", "--look", "plain",
        "--res", "240x135", "--spp", "8", "--no-panel", "--no-chart", "--no-encode")
    display(Image(filename=str(target / "beauty/frame_000000.png")))

## Load USD

The optional USD input supports polygon meshes with rigid transform animation, stage units, normals, UVs and constant Preview Surface materials. It supplies no wear images. Subdivision, deformation, skinning, instancing, per-face materials and connected textures need conversion before import. See `docs/usd.md` for the supported subset.

In [ ]:
if importlib.util.find_spec("pxr"):
    stage = root / "examples/rigid.usda"
    cli("inspect", stage)
    cli("render", stage, "--out", out / "usd", "--frames", "0:1", "--look", "plain",
        "--res", "160x90", "--spp", "4", "--no-encode")
    display(Image(filename=str(out / "usd/beauty/frame_000000.png")))
else:
    print('Install the [usd] extra to run this example.')

## Optional scalar fields

The cube animation needs no atlas or field declaration. Temperature, damage, wear and other scalar quantities
can be added independently. These generated examples use synthetic normalized images, not a heat or damage
solver. `heatmap` shows the selected quantity; `material` uses it to blend base and affected appearances.
Your own source can be installed through the `mitsuba_video.sources` entry point.

In [ ]:
for field in ("temperature", "damage"):
    source = out / field
    cli("fixture", source, "--kind", field, "--frames", "3")
    cli("render", source, "--out", out / f"{field}-render", "--field", field, "--look", "heatmap",
        "--frames", "-1:", "--res", "240x135", "--spp", "4", "--no-encode")
    display(Image(filename=str(out / f"{field}-render/frames/frame_000000.png")))

## Render settings for longer jobs

Use `-q preview`, `draft` or `paper` to choose resolution and samples, then override with `--res` and `--spp` as needed. `--variant auto` probes CUDA, Metal, LLVM and scalar backends in that order. `--variant cuda --denoise` enables guided OptiX denoising on NVIDIA; inspect fine wear detail before relying on it. `--blend-device cpu` and `gpu` choose the wear blend separately. Per-step timings are stored in `render_config.json`. Run `mitsuba-video tui` in a terminal for guided setup.